In [92]:
import pandas as pd

df = pd.read_parquet('/Users/lrome/Desktop/AI4ALL/TaxBurdenEquityAnalyzer/data/raw/cps_filers_clean.parquet')

print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

Shape: (64696, 59)

Column names:
['year', 'serial', 'month', 'cpsid', 'asecflag', 'asecwth', 'statefip', 'hhincome', 'pernum', 'cpsidp', 'cpsidv', 'asecwt', 'age', 'sex', 'race', 'marst', 'vetstat', 'famsize', 'nchild', 'nchlt5', 'ftype', 'yrimmig', 'citizen', 'nativity', 'hispan', 'empstat', 'occ2010', 'ind', 'classwkr', 'educ', 'diffany', 'wkswork1', 'uhrsworkly', 'pension', 'firmsize', 'ftotval', 'inctot', 'incwage', 'incbus', 'incss', 'incretir', 'incint', 'incdivid', 'incrent', 'ctccrd', 'actccrd', 'adjginc', 'eitcred', 'fedtaxac', 'fica', 'filestat', 'depstat', 'margtax', 'taxinc', 'spmwt', 'spmfedtaxac', 'spmftotval', 'spmfamunit', 'eff_rate']


In [91]:
import pandas as pd

df_raw = pd.read_parquet('/Users/lrome/Desktop/AI4ALL/TaxBurdenEquityAnalyzer/data/raw/cps_filers_clean.parquet')

print("Shape:", df_raw.shape)
print("View key demographic columns:")
print(df_raw[['race', 'sex', 'age', 'statefip', 'eff_rate', 'adjginc']].describe())

Shape: (64696, 59)
View key demographic columns:
             race       sex        age   statefip   eff_rate        adjginc
count     64696.0   64696.0    64696.0    64696.0    64696.0        64696.0
mean   173.646423  1.477773  45.869374  27.646609   5.165665   93169.116823
std    178.592739   0.49951   17.95193  16.346195   9.997522  120877.668906
min         100.0       1.0       15.0        1.0 -49.990633            1.0
25%         100.0       1.0       31.0       12.0   1.071485        28000.0
50%         100.0       1.0       44.0       28.0   6.624579        59150.0
75%         100.0       2.0       60.0       42.0  10.501357       116402.0
max         830.0       2.0       85.0       56.0  33.368547      2276498.0


In [88]:
import pandas as pd
df_raw = pd.read_parquet('/Users/lrome/Desktop/AI4ALL/TaxBurdenEquityAnalyzer/data/raw/cps_filers_clean.parquet')

race_labels = {
    100: 'White',
    200: 'Black',
    300: 'American Indian',
    650: 'Asian or Pacific Islander',
    651: 'Asian only',
    652: 'Hawaiian/Pacific Islander',
    700: 'Other',
    999: 'Blank'
}

df_raw['race_label'] = df_raw['race'].map(race_labels).fillna('Mixed/Other')

print("Race distribution:")
print(df_raw['race_label'].value_counts())
print("\nAs percentage:")
print(df_raw['race_label'].value_counts(normalize=True).round(3) * 100)

Race distribution:
race_label
White                        49927
Black                         7345
Asian only                    4443
Mixed/Other                   1725
American Indian                922
Hawaiian/Pacific Islander      334
Name: count, dtype: int64

As percentage:
race_label
White                        77.2
Black                        11.4
Asian only                    6.9
Mixed/Other                   2.7
American Indian               1.4
Hawaiian/Pacific Islander     0.5
Name: proportion, dtype: float64


In [102]:
# Deduction/Capital gains access by income level

print("CAPITAL GAINS CONCENTRATION")


# Create income brackets
bins = [0, 25000, 50000, 75000, 100000, 200000, float('inf')]
labels = ['<25k', '25-50k', '50-75k', '75-100k', '100-200k', '200k+']
df_raw['income_bracket'] = pd.cut(df_raw['adjginc'], bins=bins, labels=labels)

# Capital gains by bracket
cap_gains = df_raw.groupby('income_bracket', observed=True)[['incdivid', 'incrent', 'incint']].mean().round(0)
print("\nAverage investment income by bracket:")
print(cap_gains)


print("\nFILING STATUS DISTRIBUTION")
filestat_labels = {
    1: 'Joint, both <65',
    2: 'Joint, one 65+',
    3: 'Joint, both 65+',
    4: 'Head of household',
    5: 'Single',
    6: 'Nonfiler'
}
df_raw['filestat_label'] = (df_raw['filestat'].map
                            (filestat_labels))
print(df_raw['filestat_label'].value_counts())
print("\nAs percentage:")
print((df_raw['filestat_label'].value_counts(normalize=True)*100).round(1))

CAPITAL GAINS CONCENTRATION

Average investment income by bracket:
                incdivid  incrent   incint
income_bracket                            
<25k               111.0    114.0    361.0
25-50k             141.0    184.0    500.0
50-75k             315.0    338.0   1089.0
75-100k            547.0    609.0   2027.0
100-200k          1301.0   1070.0   4585.0
200k+             5638.0   7444.0  14702.0

FILING STATUS DISTRIBUTION
filestat_label
Single               32242
Joint, both <65      20219
Head of household     5583
Joint, both 65+       4673
Joint, one 65+        1979
Name: count, dtype: int64

As percentage:
filestat_label
Single               49.8
Joint, both <65      31.3
Head of household     8.6
Joint, both 65+       7.2
Joint, one 65+        3.1
Name: proportion, dtype: float64


In [105]:

print("MISSING INCOME SOURCES")
# How many rows of income data do not sum correctly?
df_raw['income_sum'] = (df_raw['incwage'] + df_raw['incbus'] +
                         df_raw['incss'] + df_raw['incretir'] +
                         df_raw['incint'] + df_raw['incdivid'] +
                         df_raw['incrent'])
mismatch = (abs(df_raw['income_sum'] - df_raw['inctot']) > 100).sum()
print(f"\nRows where income disparities do not add up: {mismatch} ({mismatch/len(df_raw)*100:.1f}%)")


print("CHILD TAX CREDIT MISMATCH")

ctc_mismatch = ((df_raw['ctccrd'] > 0) & (df_raw['nchild'] == 0)).sum()
print(f"\nRows with CTC but no children in household: {ctc_mismatch}")
print("These represent divorced parents or non-traditional households")


print(" SURVEY WEIGHT IMPACT")

unweighted = df_raw['eff_rate'].mean()
weighted = (df_raw['eff_rate'] * df_raw['asecwt']).sum() / df_raw['asecwt'].sum()
print(f"Unweighted average effective rate: {unweighted:.2f}%")
print(f"Weighted average effective rate:   {weighted:.2f}%")
print(f"Difference: {weighted - unweighted:.2f} percentage points")

MISSING INCOME SOURCES

Rows where income disparities do not add up: 7617 (11.8%)
CHILD TAX CREDIT MISMATCH

Rows with CTC but no children in household: 388
These represent divorced parents or non-traditional households
 SURVEY WEIGHT IMPACT
Unweighted average effective rate: 5.17%
Weighted average effective rate:   5.56%
Difference: 0.39 percentage points


In [93]:
# Step 1: Replace uhrsworkly 999 with (or 0) & Replace occ2010 9999 with blank or "not working".
#999 in uhrsworkly = "did not work" not actual hours.
df['uhrsworkly'] = df['uhrsworkly'].replace(999,pd.NA)
df['occ2010'] = df['occ2010'].replace(9999,pd.NA)

print("uhrsworkly 999 count:",(df['uhrsworkly'] == 999).sum())
print("occ2010 9999 count:", (df['occ2010'] == 9999).sum())

uhrsworkly 999 count: 0
occ2010 9999 count: 0


In [94]:
# Step 2: Fill incretir with 0 for every person under 58 years of age.
# File partially cleaned.
df.loc[df['age'] < 58, 'incretir'] = 0
# incretir shows 0 NaN values
print("incretir NaN count:", (df['incretir'] == 0).sum())

incretir NaN count: 61485


In [95]:
# Step 3: Handle the "0 = not asked" codes in pension, firmsize, vetstat, diffany, and nativity.
df['pension'] = df['pension'].replace(0, pd.NA)
df['firmsize'] = df['firmsize'].replace(0, pd.NA)
df['vetstat'] = df['vetstat'].replace(0,pd.NA)
df['diffany'] = df['diffany'].replace(0,pd.NA)
df['nativity'] = df['nativity'].replace(0, pd.NA)
 # Check all counts to 0
print("pension 0 count:", (df['pension'] == 0).sum())
print("firmsize 0 count:", (df['firmsize'] == 0).sum())
print("vetstat 0 count:", (df['vetstat'] == 0).sum())
print("diffany 0 count:", (df['diffany'] == 0).sum())
print("nativity 0 count:", (df['nativity'] == 0).sum())

pension 0 count: 0
firmsize 0 count: 0
vetstat 0 count: 0
diffany 0 count: 0
nativity 0 count: 0


In [96]:
# Step 4: Drop one of the two rows for household 8990 (person 10 or 12).
household_8990 = df[df['serial'] == 8990]
#Verify household 8990 has 3 rows not 4
print("Rows in household 8990:", len(household_8990))
print(household_8990[['serial', 'pernum','inctot','adjginc','fedtaxac']])
df = df.drop(index=13114)
household_8990_check = df[df['serial'] == 8990]
print("Row 12 in 8990 dropped, after drop:", len(household_8990_check))
print("New total rows:", len(df))

Rows in household 8990: 4
       serial  pernum  inctot  adjginc  fedtaxac
13103    8990       1   19600    19600     -3995
13105    8990       3   36000    36000      2438
13112    8990      10   60104    38351      1065
13114    8990      12   60104    38351      1065
Row 12 in 8990 dropped, after drop: 3
New total rows: 64695


# Negative values and capped values

Negative values in fedtaxac, inincbus, incrent, and inctot are kept, they represent real losses and refundable credits like EITC. (Earned income tax credit).


Capped values (999, 999 in incrent/incdivid, age groups at 80/85) are kept but noted as limitations. Medians should be used for these values instead of means at the top of income distribution.


In [97]:
#Step 7: For any tax-rate analysis, drop or cap rows with very small AGI.
# Rates below -50% were already removed from file.
df = df[df['adjginc'] >= 1000]

print("Rows remaining after AGI filter:", len(df))
print("Minimum AGI now:", min(df['adjginc']))

Rows remaining after AGI filter: 64335
Minimum AGI now: 1000


In [98]:
#Step 8: Rename yrimmig to years_in_us (or similar)
df = df.rename(columns={'yrimmig': 'years_in_us'})
# Renamed to years_in_us (since moving to US) not actual year.

print("'yrimmig' in columns:", 'yrimmig' in df.columns)
print("'years_in_us' in columns:", 'years_in_us' in df.columns)

'yrimmig' in columns: False
'years_in_us' in columns: True


In [99]:
#Step 9: Drop year, month, and asecflag. Select: income year 2023.
df = df.drop(columns=['year', 'month', 'asecflag'])

print("Remaining columns:", len(df.columns))
print("'year' in columns:", 'year' in df.columns)
print("'month' in columns:", 'month' in df.columns)
print("'asecflag' in columns:",'asecflag' in df.columns)

Remaining columns: 56
'year' in columns: False
'month' in columns: False
'asecflag' in columns: False


# Survey Weights

SURVEY WEIGHT DISTORTION


Unweighted rate: 5.17%

Weighted rate:   5.56%

Difference:      0.39 percentage points

Impact: Without weights model learns from
distorted picture of American taxpayers

All estimate model training should use asecwth survey weights.
Use (asecwt, and spmwt for the SPM columns) in every estimate
The weighted average effective tax rate differs from the unweighted average.
ML pass asecwth as sample_weight when training model.

In [100]:
#Step 11. Convert the column types to regular numpy types / Use float for all to handle NaN values.
df = df.astype({col: 'float64' for col in df.select_dtypes(include=['Int64', 'Float64','int64','float64']).columns})


print("Column type conversion:")
print(df.dtypes.value_counts())
print("\n Final Dataframe Shape:", df.shape)
df.to_parquet('../data/processed/cps_filers_clean_final.parquet', index=False)

Column type conversion:
float64    56
Name: count, dtype: int64

 Final Dataframe Shape: (64335, 56)
